# Walkthrough Video to 3D Gaussian Splat

Smartphone walkthrough video, in. Downloadable 3D Gaussian Splat (`.ply` + `.splat`), out. All in Colab.

**Pipeline:** frames, then Structure-from-Motion for camera poses (COLMAP / hloc), then
[Splatfacto](https://docs.nerf.studio/nerfology/methods/splat.html) optimization (Nerfstudio + `gsplat`),
then export. No pretrained weights; the splat is fit to *your* scene.

*Chosen over gsplat-standalone, DN-Splatter, OpenSplat, and raw 3DGS/2DGS for the best
reliability-per-quality: actively maintained, fully automated, and it embeds gsplat's MCMC
engine + camera-pose refinement. DN-Splatter is noted at the end as an indoor-geometry upgrade.*

> **First:** `Runtime -> Change runtime type -> GPU` (T4 works; L4/A100 are faster). Then run cells top to bottom.


## 1. GPU check
Confirms a CUDA GPU before doing anything expensive.

In [ ]:
#@title Verify GPU { display-mode: "form" }
import subprocess, sys, textwrap

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip()
    name, mem, driver = [x.strip() for x in out.split(",")]
    GPU_MEM_GB = float(mem.split()[0]) / 1024.0
    print(f"GPU     : {name}")
    print(f"VRAM    : {mem}   Driver: {driver}")
    print(f"Python  : {sys.version.split()[0]}")
    if GPU_MEM_GB < 14:
        print("\nNote: <16 GB VRAM (typical free T4). Prefer the 'balanced' preset\n"
              "and keep target_frames moderate to avoid out-of-memory.")
    else:
        print("\nPlenty of VRAM, 'high' or 'max' preset is fine.")
except Exception:
    GPU_MEM_GB = 0.0
    print(textwrap.dedent("""
        NO GPU DETECTED.
        Runtime -> Change runtime type -> Hardware accelerator -> GPU, then re-run.
        (Changing it restarts the runtime, so run this cell again after.)"""))
    raise SystemExit("Stop until a GPU is available.")

## 2. Settings

Tuned for a **T4 (16 GB VRAM)** and sharp indoor reconstruction, on **nerfstudio `main`**. Frames are
cached as `uint8` in CPU RAM at an auto-chosen resolution (RAM-safe), and `splatfacto-mcmc` caps the
Gaussian count for predictable VRAM:

- **quality_preset** , `balanced` = `splatfacto` (~1.0 M, fast) / `high` = `splatfacto-mcmc` ~1.8 M
  (recommended) / `max` = `splatfacto-mcmc` ~3.0 M (most detail; auto-reduces if VRAM is tight).
- **train_iterations** , higher = better + slower. 30k is a good default for a room/apartment.
- **target_frames** , 100 to 300 for a walkthrough. More frames = better coverage (RAM is safe now).
- **sfm_tool** , `colmap` (reliable default) or `hloc` (SuperPoint+SuperGlue, better for
  shaky / low-texture phone video).


In [ ]:
#@title Pipeline settings { display-mode: "form" }
quality_preset   = "high"     #@param ["balanced", "high", "max"]
train_iterations = 30000      #@param {type:"slider", min:5000, max:60000, step:1000}
target_frames    = 200        #@param {type:"slider", min:50, max:500, step:10}
sfm_tool         = "colmap"   #@param ["colmap", "hloc"]
scene_name       = "my_scene" #@param {type:"string"}

import re, os
scene_name = re.sub(r"[^A-Za-z0-9_\-]", "_", scene_name) or "my_scene"
WORK       = "/content/gsplat_project"
VIDEO_DIR  = f"{WORK}/video"
DATA_DIR   = f"{WORK}/processed/{scene_name}"
OUTPUT_DIR = f"{WORK}/outputs"
EXPORT_DIR = f"{WORK}/exports/{scene_name}"
for d in (VIDEO_DIR, DATA_DIR, OUTPUT_DIR, EXPORT_DIR):
    os.makedirs(d, exist_ok=True)
print(f"{quality_preset=}  {train_iterations=}  {target_frames=}  {sfm_tool=}  {scene_name=}")

## 3. System tools
Installs COLMAP + FFmpeg (~1 to 2 min).

In [ ]:
#@title Install COLMAP + FFmpeg (+ progress helpers) { display-mode: "form" }
import subprocess, sys, os, time, threading, re

def run(cmd, desc=None, log_path=None, ok_msg=None):
    """Compact runner for quick commands (installs, downloads)."""
    if desc: print(f"-> {desc}")
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    tail, logf = [], open(log_path, "w") if log_path else None
    for line in proc.stdout:
        if logf: logf.write(line)
        tail.append(line.rstrip()); tail = tail[-400:]
        sys.stdout.write("\r   " + line.rstrip()[:100].ljust(102)); sys.stdout.flush()
    proc.wait()
    if logf: logf.close()
    sys.stdout.write("\r" + " " * 106 + "\r"); sys.stdout.flush()
    if proc.returncode != 0:
        print(f"   FAILED: {desc or cmd}\n   --- last lines " + "-" * 30)
        for l in tail[-25:]: print("   " + l)
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")
    print(f"   {ok_msg or 'done'}")
    return "\n".join(tail)

def gpu_stat():
    """(util%, used_GB, total_GB) or None."""
    try:
        q = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"], text=True).strip().splitlines()[0]
        u, used, tot = [int(x) for x in q.split(",")]
        return u, used / 1024.0, tot / 1024.0
    except Exception:
        return None

def mem_used():
    """(used_GB, total_GB) system RAM, or None. Used = MemTotal - MemAvailable."""
    try:
        info = {}
        for l in open("/proc/meminfo"):
            parts = l.split(":")
            if len(parts) == 2: info[parts[0]] = int(parts[1].split()[0])
        tot = info["MemTotal"] / 1e6
        return tot - info["MemAvailable"] / 1e6, tot
    except Exception:
        return None

def run_live(cmd, log_path, total_steps=None, stage_map=None):
    """Run a long command with a live progress bar (total_steps) and/or a stage
    heartbeat (stage_map). New stages are printed with a [+MM:SS] timestamp so you
    can see exactly where startup time goes. Saves the log; returns (rc, tail)."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    st = {"step": 0, "stage": "starting", "t0": time.time(), "alive": True}
    seen, tail = set(), []
    def stamp(name):
        el = int(time.time() - st["t0"]); mm, ss = divmod(el, 60)
        m = mem_used(); ram = f" RAM {m[0]:.1f}/{m[1]:.0f}GB " if m else "  "
        sys.stdout.write("\r" + " " * 122 + "\r")
        print(f"  [+{mm:d}m{ss:02d}s]{ram}{name}")
    peak = {"ram": 0.0}
    def heartbeat():
        n = 0
        while st["alive"]:
            el = int(time.time() - st["t0"]); mm, ss = divmod(el, 60)
            g = gpu_stat(); m = mem_used()
            if m and m[0] > peak["ram"]: peak["ram"] = m[0]
            gs = f"GPU {g[0]:>3d}% {g[1]:.1f}/{g[2]:.0f}GB" if g else "GPU n/a"
            rs = f"RAM {m[0]:.1f}/{m[1]:.0f}GB" if m else "RAM n/a"
            if total_steps and st["step"] > 0:
                pct = min(st["step"] / total_steps, 1.0); fill = int(pct * 18)
                line = (f"[{'#'*fill}{'.'*(18-fill)}] {pct*100:5.1f}% "
                        f"{st['step']}/{total_steps} | {gs} | {rs} | {mm}m{ss:02d}s")
            else:
                line = f"{st['stage']:<20} | {gs} | {rs} | {mm}m{ss:02d}s"
            sys.stdout.write("\r" + line[:120].ljust(122)); sys.stdout.flush()
            n += 1
            if n % 4 == 0 and m:   # persist a RAM trajectory line every ~16s (survives copy-paste)
                sys.stdout.write("\r" + " " * 122 + "\r")
                print(f"  [+{mm:d}m{ss:02d}s] RAM {m[0]:.1f}/{m[1]:.0f}GB "
                      f"(peak {peak['ram']:.1f}) | {st['stage']}")
            time.sleep(4)
    th = threading.Thread(target=heartbeat, daemon=True); th.start()
    step_re = re.compile(r"(\d+)\s*\(\s*[\d.]+\s*%\)")   # nerfstudio "1200 (4.00%)"
    with open(log_path, "w") as lf:
        for line in proc.stdout:
            lf.write(line); s = line.rstrip(); tail.append(s); tail = tail[-500:]
            if stage_map:
                low = s.lower()
                for kw, name in stage_map.items():
                    if kw in low:
                        st["stage"] = name
                        if name not in seen: seen.add(name); stamp(name)
            if total_steps:
                m = step_re.search(s)
                if m and int(m.group(1)) <= total_steps:
                    if "training loop" not in seen:
                        seen.add("training loop"); stamp("training loop started")
                    st["step"] = int(m.group(1))
            if any(k in s for k in ("Traceback", "CUDA out of memory", "Killed",
                                    "ERROR", "Invalid", "Unrecognized", "core dumped")):
                sys.stdout.write("\r" + " " * 122 + "\r"); print("  " + s[:140])
    proc.wait(); st["alive"] = False; th.join(timeout=1)
    sys.stdout.write("\r" + " " * 122 + "\r"); sys.stdout.flush()
    return proc.returncode, "\n".join(tail[-45:])

run("apt-get -qq update", "apt update")
run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y colmap ffmpeg xvfb",
    "colmap + ffmpeg + xvfb")
for tool in ("colmap", "ffmpeg"):
    v = subprocess.run([tool, "-h" if tool == "colmap" else "-version"],
                       capture_output=True, text=True)
    print(f"   {tool}: {(v.stdout or v.stderr).splitlines()[0][:55]}")
print("System tools ready.")

## 4. Install Nerfstudio (latest `main`) + gsplat

Installs nerfstudio from **`main`** (not the 2024 release), mainly for **`splatfacto-mcmc`**
(fixed Gaussian budget -> bounded VRAM and clean surfaces), plus newer gsplat and rasterizer options.

Heaviest step (~6 to 10 min). `main` pins **torch 2.7.1 + gsplat 1.4.0**, so pip will reinstall
torch , that's expected. We build gsplat for the T4 only (`sm_75`) and warm it up to catch issues
early. tiny-cuda-nn is skipped (Splatfacto doesn't need it).


In [ ]:
#@title Install Nerfstudio (main) + gsplat { display-mode: "form" }
import os, subprocess, sys

# This project always runs on a T4 (compute capability 7.5). Set the build arch
# BEFORE installing, and do NOT import torch here (main reinstalls torch 2.7.1).
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

nerfstudio_ref = "main"  #@param {type:"string"}
print(f"Installing nerfstudio @ {nerfstudio_ref} (pulls torch 2.7.1 + gsplat 1.4.0).")
print("Heaviest step (~6 to 10 min); pip will reinstall torch , this is expected.\n")
run(f"pip -q install 'git+https://github.com/nerfstudio-project/nerfstudio.git@{nerfstudio_ref}'",
    "nerfstudio (main) + gsplat + deps", log_path="/content/install_nerfstudio.log")

print("\nWarming up gsplat CUDA kernels (first build can take a few minutes)...")
warm = subprocess.run(
    [sys.executable, "-c",
     "import warnings; warnings.filterwarnings('ignore');"
     "import torch, gsplat; from gsplat import rasterization;"
     "print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gsplat', gsplat.__version__)"],
    capture_output=True, text=True)
print("  " + (warm.stdout.strip() or warm.stderr[-1200:]))
if warm.returncode != 0:
    print("  If this is a build error, re-run this cell once (wheels/kernels cache), or")
    print("  Runtime -> Restart runtime, then run cells 1-4 again.")

cli = subprocess.run(["ns-train", "--help"], capture_output=True, text=True)
if cli.returncode != 0:
    raise RuntimeError("ns-train not on PATH, check /content/install_nerfstudio.log")
# Confirm the main-only features we rely on are present.
mcmc = subprocess.run(["ns-train", "splatfacto-mcmc", "--help"], capture_output=True, text=True)
print("Nerfstudio CLI ready | splatfacto-mcmc:",
      "yes" if mcmc.returncode == 0 else "NO (check install)")

## 5. Provide your video

Pick **one** `input_method`:
- **`upload`** , a "Choose Files" button appears below; select the video.
- **`google_drive_link`** , paste a Drive **share link** (shared "Anyone with the link");
  downloaded via `gdown`.
- **`direct_url_or_path`** , a direct `https://...` video URL, or a local/Drive-mounted file path.

Types: `.mp4 .mov .m4v .avi .mkv .webm`. Capture tip: walk slowly, good light, overlap views, 30 to 90 s.


In [ ]:
#@title Get the video: upload / Drive link / URL or path { display-mode: "form" }
import os, sys, shutil, subprocess, json
input_method = "upload"  #@param ["upload", "google_drive_link", "direct_url_or_path"]
#@markdown Paste here for google_drive_link / direct_url_or_path (leave blank for upload):
link_or_url = ""  #@param {type:"string"}

# To use a Google Drive *path*: uncomment, run, then pick direct_url_or_path + set the path.
# from google.colab import drive; drive.mount('/content/drive')

VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")
VIDEO_PATH = None

if input_method == "upload":
    from google.colab import files
    print("Click 'Choose Files' and pick your walkthrough video...\n")
    up = files.upload()
    if not up: raise RuntimeError("No file selected. Re-run and choose a video.")
    fn = list(up.keys())[0]
    if not fn.lower().endswith(VIDEO_EXTS):
        raise ValueError(f"'{fn}' is not a video type {VIDEO_EXTS}.")
    VIDEO_PATH = os.path.join(VIDEO_DIR, fn)
    open(VIDEO_PATH, "wb").write(up[fn])
else:
    src = link_or_url.strip()
    if not src:
        raise ValueError("link_or_url is empty (or set input_method='upload').")
    if input_method == "google_drive_link" or "drive.google.com" in src or "docs.google.com" in src:
        try: import gdown
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gdown"]); import gdown
        print("Downloading from Google Drive...\n")
        got = gdown.download(url=src, output=os.path.join(VIDEO_DIR, "walkthrough_input"),
                             quiet=False, fuzzy=True)
        if not got or not os.path.isfile(got):
            raise RuntimeError("gdown failed, ensure the link is shared 'Anyone with the link' "
                               "and points to a single file (not a folder).")
        VIDEO_PATH = got
    elif src.startswith(("http://", "https://")):
        ext = os.path.splitext(src.split("?")[0])[1].lower() or ".mp4"
        VIDEO_PATH = os.path.join(VIDEO_DIR, "walkthrough_input" + ext)
        run(f"wget -q --show-progress -O '{VIDEO_PATH}' '{src}'", "Downloading video")
        if os.path.getsize(VIDEO_PATH) < 1024:
            raise RuntimeError("Downloaded file is tiny/empty, URL may not be a direct video link.")
    elif os.path.isfile(src):
        VIDEO_PATH = os.path.join(VIDEO_DIR, os.path.basename(src)); shutil.copy(src, VIDEO_PATH)
    else:
        raise FileNotFoundError(f"'{src}' is neither a URL nor an existing file path.")

# Quick probe for immediate feedback.
r = subprocess.run("ffprobe -v error -select_streams v:0 -show_entries "
                   "stream=width,height:format=duration -of json " + f"'{VIDEO_PATH}'",
                   shell=True, capture_output=True, text=True)
print(f"\nVideo ready: {os.path.basename(VIDEO_PATH)}  "
      f"({os.path.getsize(VIDEO_PATH)/1e6:.1f} MB)")
try:
    j = json.loads(r.stdout); st = j["streams"][0]
    dur = float(j["format"]["duration"])
    print(f"  {st['width']}x{st['height']}, {dur:.1f} s")
    if dur < 5: print("  WARNING: very short (<5 s), reconstruction may be poor.")
except Exception:
    print("  (details unavailable, but the file will still be processed)")

## 6. Recover camera poses (Structure-from-Motion)

The modern replacement for manual calibration: `ns-process-data` extracts frames and runs SfM to
recover per-frame camera poses + a sparse point cloud (used to initialize the Gaussians).
Usually the **slowest** stage. COLMAP runs on **CPU** (`--no-gpu`) because GPU SIFT crashes on
headless Colab; `hloc` keeps the GPU. We verify a valid `transforms.json` at the end.


In [ ]:
#@title Run SfM { display-mode: "form" }
import os, json, glob, shutil
if os.path.isdir(DATA_DIR) and os.listdir(DATA_DIR):
    shutil.rmtree(DATA_DIR); os.makedirs(DATA_DIR, exist_ok=True)

# On headless Colab, COLMAP's GPU SIFT crashes (needs an OpenGL context, and is
# extra fragile after main reinstalled torch/CUDA), so run COLMAP feature
# extraction + matching on CPU via --no-gpu. Reliable; a few minutes for ~150
# frames. hloc keeps the GPU (it uses its own torch-based matcher).
# (--verbose is omitted: on failure nerfstudio's verbose path crashes decoding
#  a None stderr, which hides the real error.)
match    = "sequential" if sfm_tool == "colmap" else "exhaustive"
gpu_flag = "--no-gpu" if sfm_tool == "colmap" else ""
cmd = (f"xvfb-run -a ns-process-data video --data '{VIDEO_PATH}' --output-dir '{DATA_DIR}' "
       f"--num-frames-target {target_frames} --sfm-tool {sfm_tool} "
       f"--matching-method {match} {gpu_flag}").strip()
print(cmd, "\n")
# COLMAP stages are long and mostly silent; show which stage is running + elapsed.
stage_map = {
    "extract": "extracting frames", "copy": "copying images",
    "feature": "extracting features", "matching": "matching features",
    "match": "matching features", "reconstruct": "sparse mapping",
    "mapper": "sparse mapping", "bundle": "bundle adjustment",
    "undistort": "undistorting", "refine": "refining poses",
    "done": "finalizing",
}
print("Running SfM on CPU (this is the slow stage). Full log: /content/ns_process.log\n")
rc, tail = run_live(cmd, "/content/ns_process.log", stage_map=stage_map)
if rc != 0:
    print("\nSfM failed. Try sfm_tool='hloc' (cell 2), recapture with slower motion / more")
    print("overlap, or raise target_frames.\n--- log tail ---\n", tail)
    raise RuntimeError("SfM failed. See /content/ns_process.log")
print("SfM finished.")

tj = os.path.join(DATA_DIR, "transforms.json")
if not os.path.isfile(tj):
    raise RuntimeError("No transforms.json, SfM produced no valid poses.")
T = json.load(open(tj))
n_frames = len(T.get("frames", []))
n_imgs = len(glob.glob(os.path.join(DATA_DIR, "images", "*")))
has_pts = os.path.isfile(os.path.join(DATA_DIR, "sparse_pc.ply")) or bool(
    glob.glob(os.path.join(DATA_DIR, "colmap", "sparse", "**", "points3D*"), recursive=True))
print(f"\nregistered frames: {n_frames} / {n_imgs} images   sparse points: {'yes' if has_pts else 'no'}")
if n_frames < 20:
    print("WARNING: <20 frames registered, quality will be low. Try sfm_tool='hloc' or recapture.")
else:
    print("Poses look good. Ready to train.")

## 7. Train the Gaussian Splat (nerfstudio `main`, T4-tuned)

Fits Gaussians to your images (no pretrained weights), using `main`'s features that fit the T4:

- **`splatfacto-mcmc`** (high/max) , holds a **fixed Gaussian budget** (`max-gs-num`), so VRAM is
  predictable and surfaces come out clean with few floaters (ideal for walls/floors). `balanced`
  uses plain `splatfacto`.
- **System-RAM `Killed` at startup** , the OOM is **resolution-independent** (it happened even at 1/8
  resolution / 0.05 GB of images), so it was never image data. It's unbounded **native-library thread
  buffers** (OpenBLAS/OpenMP/OpenCV) allocated during model init , we cap every threading lib to 2 and
  bound `max-thread-workers` (the caps under which an earlier run succeeded at 4-7 GB).
- **~30-min pre-training stall** , nerfstudio wraps `get_viewmat` in `@torch_compile`; its first call
  compiles via inductor/max-autotune. **`TORCHDYNAMO_DISABLE=1`** skips it (runs eager, trivial op).
- **`rasterize-mode classic`** (web-viewer-compatible `.ply`) + camera-pose refinement +
  bilateral-grid exposure, on `cache-images cpu`/`uint8`.

A **live bar** shows step% + **GPU + system RAM** + elapsed, and each startup stage prints a `[+MM:SS]`
stamp **with the RAM level** , so any spike is pinpointed. CUDA (VRAM) OOM shrinks the Gaussian budget;
resolution is never reduced (it's irrelevant to the OOM).


In [ ]:
#@title Train (nerfstudio main) { display-mode: "form" }
import os, glob
from PIL import Image

# TWO INDEPENDENT ROOT CAUSES, fixed here:
# (1) System-RAM OOM ("Killed") during startup. It is resolution-INDEPENDENT (it
#     happened even at 1/8 res / 0.05 GB of images), so it is NOT image data , it
#     is unbounded native-library thread buffers (OpenBLAS/OpenMP/OpenCV) allocated
#     during model init. Capping every threading lib keeps startup RAM bounded
#     (this is what made an earlier run succeed at 4-7 GB).
# (2) ~30-min pre-training stall = torch.compile compiling get_viewmat on first use
#     (inductor/max-autotune). Disable it; get_viewmat runs eager (trivial op).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "2"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
print("env: os.cpu_count() =", os.cpu_count(),
      "| threading capped to 2 | torch.compile disabled")

# splatfacto-mcmc: fixed gaussian budget -> predictable T4 VRAM + clean surfaces.
BUDGET = {"balanced": 1_000_000, "high": 1_800_000, "max": 3_000_000}
gs_cap = BUDGET[quality_preset]
method = "splatfacto" if quality_preset == "balanced" else "splatfacto-mcmc"

# --- Pick a training resolution (for SPEED/VRAM, not RAM) ---------------------
# The image cache is tiny and NOT the OOM cause (proven: OOM happened even at 1/8
# res). We just cap the long side at ~1600 px , the splatting quality sweet spot ,
# so training isn't needlessly slow on huge frames.
cache_mode = "disk-stream"  #@param ["disk-stream", "cpu"]
#@markdown `disk-stream` (default) streams frames from disk with 0 dataloader
#@markdown workers , it uses nerfstudio's ImageBatchStream path, which BYPASSES the
#@markdown bulk "Caching / undistorting train images" step where cpu mode was
#@markdown OOM-killed. `cpu` caches all frames in RAM (fine if you have headroom).

imgs = sorted(glob.glob(os.path.join(DATA_DIR, "images", "*")))
if not imgs:
    raise RuntimeError("No images in the dataset , re-run the SfM cell first.")
N = len(imgs); W, H = Image.open(imgs[0]).size
factor = 1
while max(W, H) / factor > 1600: factor *= 2

# --- Startup diagnostics: is the OOM baseline memory or a real allocation? -----
import json as _json
_m = mem_used()
if _m: print(f"Baseline RAM (before ns-train): {_m[0]:.1f}/{_m[1]:.0f} GB used")
try:
    _tj = _json.load(open(os.path.join(DATA_DIR, "transforms.json")))
    print(f"Dataset frames (undistorted at startup): {len(_tj.get('frames', []))}")
except Exception:
    pass
_ply = os.path.join(DATA_DIR, "sparse_pc.ply")
if os.path.isfile(_ply):
    _npts = None
    with open(_ply, "rb") as _f:
        for _ in range(40):
            _l = _f.readline().decode("latin1", "ignore")
            if _l.startswith("element vertex"): _npts = int(_l.split()[-1])
            if _l.startswith("end_header"): break
    print(f"Sparse point cloud: {_npts} points, {os.path.getsize(_ply)/1e6:.1f} MB")
print(f"Images: {N} @ {W}x{H} -> training at 1/{factor} ({W//factor}x{H//factor}), "
      f"cache={cache_mode}\n")

# uint8 CPU cache = no spawn workers, low RAM. All flags exist in nerfstudio main.
common = [
    f"--data '{DATA_DIR}'", f"--output-dir '{OUTPUT_DIR}'",
    f"--max-num-iterations {train_iterations}",
    "--viewer.quit-on-train-completion True", "--vis tensorboard",
    f"--experiment-name {scene_name}",
    "--pipeline.datamanager.max-thread-workers 2",   # bound undistortion concurrency
    "--pipeline.model.camera-optimizer.mode SO3xR3",
    "--pipeline.model.use-bilateral-grid True",
    "--pipeline.model.rasterize-mode classic",   # exported .ply works in web viewers
]
if cache_mode == "disk-stream":
    # disk keeps frames off RAM; 0 workers avoids main's spawn-a-torch-per-worker.
    cache_flags = ["--pipeline.datamanager.cache-images disk",
                   "--pipeline.datamanager.dataloader-num-workers 0"]
else:
    cache_flags = ["--pipeline.datamanager.cache-images cpu",
                   "--pipeline.datamanager.cache-images-type uint8"]
common += cache_flags
def model_flags(cap):
    if method == "splatfacto-mcmc":
        return [f"--pipeline.model.max-gs-num {cap}"]
    return ["--pipeline.model.use-scale-regularization True"]

def build(cap, f):
    # Dataparser args (--downscale-factor) go AFTER the nerfstudio-data subcommand.
    return (f"ns-train {method} " + " ".join(common + model_flags(cap)) +
            f" nerfstudio-data --data '{DATA_DIR}' --downscale-factor {f}")

# Startup-stage labels so you can see where any pre-training time goes.
TRAIN_STAGES = {
    "caching / undistorting train": "undistorting train images",
    "caching / undistorting eval": "undistorting eval images",
    "loading nerfstudio": "loading dataset",
    "avoid opening images": "loading dataset",
}
cap = gs_cap
print(f"Training [{quality_preset}] -> {method}"
      + (f", ~{cap/1e6:.1f}M gaussians" if method == "splatfacto-mcmc" else "")
      + f", {train_iterations} iters. Each stage prints RAM so you can see spikes:\n")
rc, tail = run_live(build(cap, factor), "/content/ns_train.log",
                    total_steps=train_iterations, stage_map=TRAIN_STAGES)

# Single OOM fallback: CUDA VRAM -> smaller gaussian budget. (System-RAM 'Killed'
# is handled up front by the thread caps; resolution is irrelevant, so we do NOT
# downscale. If it still OOMs on RAM, the per-stage RAM stamps above pinpoint it.)
if rc != 0 and "out of memory" in tail.lower() and method == "splatfacto-mcmc":
    cap = int(gs_cap * 0.55)
    print(f"\nCUDA out of memory; retrying with ~{cap/1e6:.1f}M gaussians...\n")
    rc, tail = run_live(build(cap, factor), "/content/ns_train.log",
                        total_steps=train_iterations, stage_map=TRAIN_STAGES)

if rc != 0:
    if "Killed" in tail:
        print("\nProcess OOM-killed (system RAM). The [+MM:SS] RAM stamps above show the")
        print("stage where RAM spiked. If it spiked in a fixed startup phase, lower the")
        print("gaussian budget ('balanced' preset) or report the stamps.")
    print("\n--- log tail ---\n", tail)
    raise RuntimeError("Training failed. See /content/ns_train.log")

cfgs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "config.yml"), recursive=True),
              key=os.path.getmtime)
if not cfgs: raise RuntimeError("Training finished but no config.yml found.")
CONFIG_YML = cfgs[-1]
print("\nTraining complete.\nconfig:", CONFIG_YML)

## 8. Export
Lossless **`.ply`** (best quality; SuperSplat / PlayCanvas / Three.js) + a compact **`.splat`** for web viewers.
Runs the exporter with a `torch.load(weights_only=False)` patch , required because PyTorch 2.6+ defaults
to `weights_only=True`, which rejects the numpy objects saved in a nerfstudio checkpoint.

In [ ]:
#@title Export .ply (+ .splat) { display-mode: "form" }
import os, glob, numpy as np

# PyTorch 2.6+ defaults torch.load(weights_only=True), which rejects the numpy
# objects in a nerfstudio checkpoint -> ns-export crashes. We patch torch.load to
# weights_only=False (our own trusted checkpoint) and run the exporter in-process.
# TORCHDYNAMO_DISABLE avoids any torch.compile during the export's model setup.
export_py = r"""
import sys, warnings, torch
warnings.filterwarnings("ignore")
_orig_load = torch.load
def _load(*a, **k):
    k["weights_only"] = False
    return _orig_load(*a, **k)
torch.load = _load
cfg, out = sys.argv[1], sys.argv[2]
sys.argv = ["ns-export", "gaussian-splat", "--load-config", cfg, "--output-dir", out]
from nerfstudio.scripts.exporter import entrypoint
entrypoint()
"""
with open("/content/_export_splat.py", "w") as f:
    f.write(export_py)
run(f"TORCHDYNAMO_DISABLE=1 python /content/_export_splat.py '{CONFIG_YML}' '{EXPORT_DIR}'",
    "Exporting Gaussian splat (.ply)", log_path="/content/ns_export.log")
plys = glob.glob(os.path.join(EXPORT_DIR, "*.ply"))
if not plys: raise RuntimeError("No .ply produced, see /content/ns_export.log")
PLY_PATH = plys[0]
print(f"  .ply: {PLY_PATH}  ({os.path.getsize(PLY_PATH)/1e6:.1f} MB)")

def ply_to_splat(ply_path, splat_path):
    # antimatter15 .splat: pos(3 f32) scale(3 f32) rgba(4 u8) rot-quat(4 u8) = 32 bytes.
    from plyfile import PlyData
    v = PlyData.read(ply_path)["vertex"]
    xyz = np.stack([v["x"], v["y"], v["z"]], 1).astype(np.float32)
    scales = np.exp(np.stack([v["scale_0"], v["scale_1"], v["scale_2"]], 1).astype(np.float32))
    rots = np.stack([v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]], 1).astype(np.float32)
    rots /= (np.linalg.norm(rots, axis=1, keepdims=True) + 1e-9)
    C0 = 0.28209479177387814
    color = np.clip(0.5 + C0 * np.stack([v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]], 1), 0, 1)
    opacity = 1 / (1 + np.exp(-np.asarray(v["opacity"])))
    rgba = np.clip(np.concatenate([color, opacity[:, None]], 1) * 255, 0, 255).astype(np.uint8)
    rot_u8 = np.clip(rots * 128 + 128, 0, 255).astype(np.uint8)
    order = np.argsort(-(opacity * np.prod(scales, 1)))  # important splats first
    with open(splat_path, "wb") as f:
        for i in order:
            f.write(xyz[i].tobytes()); f.write(scales[i].tobytes())
            f.write(rgba[i].tobytes()); f.write(rot_u8[i].tobytes())
    return len(order)

SPLAT_PATH = os.path.join(EXPORT_DIR, f"{scene_name}.splat")
try:
    n = ply_to_splat(PLY_PATH, SPLAT_PATH)
    print(f"  .splat: {SPLAT_PATH}  ({n:,} gaussians, {os.path.getsize(SPLAT_PATH)/1e6:.1f} MB)")
except Exception as e:
    SPLAT_PATH = None; print("  (.splat skipped:", e, ")")

## 9. Download
Zips the results and downloads. View instantly (no install) at
[superspl.at/editor](https://superspl.at/editor) , drag in the `.ply`.

In [ ]:
#@title Zip + download { display-mode: "form" }
import os, zipfile
zip_path = os.path.join(WORK, f"{scene_name}_gaussian_splat.zip")
items = [p for p in (PLY_PATH, SPLAT_PATH) if p and os.path.isfile(p)]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in items: z.write(p, arcname=os.path.basename(p))
    z.writestr("README.txt",
               f"Scene: {scene_name}\nPipeline: Nerfstudio Splatfacto "
               f"({quality_preset}, {train_iterations} iters, sfm={sfm_tool})\n"
               "*.ply = full quality (recommended); *.splat = compact web view.\n"
               "View: https://superspl.at/editor\n")
for p in items: print(f"  {os.path.basename(p):36s} {os.path.getsize(p)/1e6:7.1f} MB")
print(f"zip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print("Auto-download unavailable:", e, "\nDownload manually from the Files panel:", zip_path)

## Troubleshooting

| Symptom | Fix |
|---|---|
| No GPU (cell 1) | Runtime -> Change runtime type -> GPU, re-run. |
| gsplat compile error (cell 4) | Re-run cell 4 once (kernels cache), or restart runtime + run 1 to 4. |
| Few/no frames in SfM (cell 6) | `sfm_tool='hloc'`; recapture slower, more overlap. |
| `Killed` / out of memory (cell 7) | Auto-retries downscaled. If it still fails: `balanced` preset, lower `target_frames`, or L4/A100. (`Killed` = system-RAM OOM, not GPU.) |
| Noisy / floaty splat | More `train_iterations`, `high`/`max` preset, more overlapping views. |

**Indoor-geometry upgrade , [DN-Splatter](https://github.com/maturk/dn-splatter):** adds depth+normal
priors on top of nerfstudio and reuses the exact `DATA_DIR` from cell 6. Heavier install / more
version-sensitive, which is why the default is Splatfacto.

**Credits:** Nerfstudio (Tancik et al.), gsplat (Ye et al.), 3D Gaussian Splatting (Kerbl et al.,
SIGGRAPH 2023), COLMAP (Schönberger & Frahm), DN-Splatter (Turkulainen et al., WACV 2025).
